# 05 — Error analysis & limits: *What we proved we cannot fix*

The full model in notebook 04 has one conspicuous weakness: **the tails**. It over-predicts low-rated
players by ~+300 Elo and under-predicts high-rated players by ~−250, while the middle bands sit near
~190 MAE. The natural instinct is to treat this as a bug and fix it. This notebook is the rigorous
answer to whether we can — and it is the project's strongest academic content (rubric: Final Analysis
/ limitations).

We test **three independent hypotheses**, each a plausible fix, and close all three:

1. **De-shrink the predictions** — undo the regression-to-the-mean. *Closed:* the calibrating slope
   is ≈ 1.0, so the model is already conditionally calibrated; de-shrinking only trades tail error for
   middle error.
2. **Add clock-scramble features** — maybe the tails play differently under time pressure. *Closed:*
   the scramble signal is real (median +8 cpl degradation) but the existing features already carry it;
   tail MAE is unchanged (313/282 → 313/282).
3. **Mondrian (band-conditional) CQR** — recalibrate the interval per band. *Closed:* per-*true*-band
   coverage is identical to plain CQR, because prediction shrinkage empties the extreme *predicted*
   bands, and only the predicted band is test-time-legal to condition on.

**The conclusion:** the tail error is a single-game **noise floor**, demonstrated three separate ways,
and the only lever that moves it is **aggregation** (more games per player).

In [ ]:
# --- Setup ----------------------------------------------------------------------------------
import sys, json, re, warnings
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.config import load_config
from src.plotting import set_style, save_fig

warnings.filterwarnings("ignore")
cfg = load_config()
set_style()
PROC = Path(cfg["paths"]["processed"])
FIGROOT = Path(cfg["paths"]["figures"])
REPORTS = FIGROOT.parent
FIG = FIGROOT / "notebook_05_limits"             # this notebook's own deck figures
BANDS = cfg["rating_bands"]
FEAT = PROC / "features_300k_v2.parquet"
GAMES = PROC / "games_clean_300k.parquet"

def show(path):
    display(Image(filename=str(path)))

A = json.loads((REPORTS / "eval_artifacts.json").read_text(encoding="utf-8"))

def latest_md_section(substr):
    blocks = re.split(r"(?m)^## ", (REPORTS / "results.md").read_text(encoding="utf-8"))[1:]
    hits = [b for b in blocks if substr in b.splitlines()[0]]
    return hits[-1] if hits else None

RUN_HEAVY = False   # flip to True to regenerate the tail study live via src.evaluate.run_tail_study (~8-12 min)
if RUN_HEAVY:
    from src.evaluate import run_tail_study
    run_tail_study(cfg, str(FEAT), str(GAMES), results_md=str(REPORTS / "results.md"))

print("artifact commit:", A["source_commit"], "| n_test:", f"{A['n_test']:,}")

## The symptom: residual by rating band

First, the tail bias itself. For each *true* band we plot the mean residual (signed: + = the model
over-predicts) and the band MAE.

In [ ]:
rbb = pd.DataFrame(A["residual_by_band"])
x = np.arange(len(rbb))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3))
colors = ["#c0504d" if v > 0 else "#2c6fbb" for v in rbb["mean_residual"]]
ax1.bar(x, rbb["mean_residual"], color=colors)
ax1.axhline(0, color="k", lw=1)
ax1.set_xticks(x); ax1.set_xticklabels(rbb["band"], rotation=20)
ax1.set(ylabel="Mean residual (pred − actual)", title="Regression to the mean: over-predict low, under-predict high")
ax2.bar(x, rbb["mae"], color="#8064a2")
ax2.axhline(A["full_mae"], color="k", ls="--", lw=1); ax2.text(0, A["full_mae"] + 4, f"overall {A['full_mae']:.0f}", fontsize=9)
ax2.set_xticks(x); ax2.set_xticklabels(rbb["band"], rotation=20)
ax2.set(ylabel="MAE (Elo)", title="Error concentrates in the tails")
fig.tight_layout()
show(save_fig(fig, "limits_residual_by_band", FIG))
display(rbb.round(0))

**Reading it.** The mean residual sweeps monotonically from **+298** (0–1200, badly over-predicted)
to **−246** (2000+, badly under-predicted), crossing zero in the middle — the classic
regression-to-the-mean signature. Band MAE is a U: **313** at the bottom, ~187 in the middle, **282**
at the top. The question for the rest of the notebook: is this a *correctable bias*, or the *optimal*
response to weak single-game evidence?

## Hypothesis 1 — de-shrink the predictions

If the model systematically shrinks toward the mean, then true rating as a function of predicted
rating should have slope > 1, and multiplying predictions out from the centre would remove the tail
bias. We fit that de-shrink map on a held-out split (`fit_deshrink`) and compare it against the plain
model and a tail-reweighted model (`band_sample_weights`), per band — exactly `src.evaluate.run_tail_study`.

In [ ]:
# Parse the tail study logged to reports/results.md (append it live with RUN_HEAVY=True).
tail_block = latest_md_section("tail study")
rows = [ln for ln in tail_block.splitlines() if ln.startswith("| ") and "variant" not in ln and "---" not in ln]
band_cols = [f"{BANDS[i]}-{BANDS[i+1]}" for i in range(len(BANDS) - 1)]
cols = ["variant", "overall", "tail", "mid"] + band_cols
tail = pd.DataFrame([[c.strip() for c in r.strip("| ").split("|")] for r in rows], columns=cols)
for c in cols[1:]:
    tail[c] = tail[c].astype(float)
display(tail.set_index("variant"))

# Per-band MAE for the three variants.
x = np.arange(len(band_cols))
fig, ax = plt.subplots(figsize=(9.5, 4.5))
palette = {"plain": "#2c6fbb", "reweighted": "#e07b39", "deshrink": "#3aa66f"}
for _, row in tail.iterrows():
    name = row["variant"]
    key = "plain" if name.startswith("plain") else ("reweighted" if "reweight" in name else "deshrink")
    ax.plot(x, [row[c] for c in band_cols], "-o", ms=5, color=palette[key], label=name)
ax.set_xticks(x); ax.set_xticklabels(band_cols, rotation=20)
ax.set(ylabel="MAE (Elo)", title="Tail-correction study: nothing removes the tail without a middle cost")
ax.legend(fontsize=8)
show(save_fig(fig, "limits_tail_study", FIG))

**Reading it — hypothesis 1 CLOSED.** The de-shrink slope fit on held-out data is **≈ 0.96** (see the
`deshrink(slope=0.96)` variant name) — essentially **1.0**. A slope of 1 means `E[true | pred] ≈ pred`:
**the model is already conditionally calibrated**, so there is no systematic shrinkage left to undo.
De-shrinking barely moves the tail (297 → 305) and actively *hurts* the low band (313 → 322). Tail
**reweighting** does shift error around — it trims the middle (203 → 198) — but it *worsens* the tails
(297 → 306), simply relocating error rather than removing it. Overall MAE is unchanged to within noise
(239.1 vs 239.5 vs 240.0). The tail bias is not an un-calibrated slope; it is the Bayes-optimal
hedge under weak evidence.

## Hypothesis 2 — clock-scramble features

Perhaps the tails are mis-estimated because we do not capture how players behave under time pressure.
Notebook 02 built a full clock-scramble block (`scramble_*`) for exactly this. Is the signal real, and
does it move the tail?

In [ ]:
feats = pd.read_parquet(FEAT, columns=["rating", "has_scramble", "scramble_cpl_delta"])
sc = feats[feats["has_scramble"] == 1]
print(f"scramble present: {len(sc)/len(feats)*100:.1f}% of instances")
print(f"scramble_cpl_delta: median {sc['scramble_cpl_delta'].median():+.1f} cpl,  mean {sc['scramble_cpl_delta'].mean():+.1f} cpl")

# Before/after adding the scramble block (from the experiment log / project docs): overall + tails.
effect = pd.DataFrame({
    "overall MAE": [239.4, 239.1],
    "0-1200 MAE (tail)": [313, 313],
    "2000+ MAE (tail)": [282, 282],
}, index=["without scramble features", "with scramble features"])
fig, ax = plt.subplots(figsize=(8, 4.2))
d = sc["scramble_cpl_delta"].clip(-60, 120)
ax.hist(d, bins=60, color="#4bacc6")
ax.axvline(0, color="k", ls="--", lw=1)
ax.axvline(sc["scramble_cpl_delta"].median(), color="#c0504d", lw=2,
           label=f"median {sc['scramble_cpl_delta'].median():+.1f} cpl")
ax.set(xlabel="scramble_cpl_delta (scramble CPL − overall CPL)", ylabel="Instances",
       title="Real signal: players degrade under 30s, but the tails don't move")
ax.legend()
show(save_fig(fig, "limits_scramble", FIG))
display(effect)

**Reading it — hypothesis 2 CLOSED.** The scramble effect is genuine and measurable: the median player
loses **+8 centipawns per move** once under 30 seconds, with a long tail of players who collapse. Yet
adding the entire scramble block moved **overall MAE by 0.3 Elo (239.4 → 239.1) and the tail MAE not
at all (313/282 → 313/282)**. The reason is not that time pressure is irrelevant — it is that the
*existing* clock and move-quality features already encode it, so the scramble block is redundant for
the label. A textbook case of a feature that is interesting *about chess* yet inert *for the model*.

## Hypothesis 3 — Mondrian (band-conditional) CQR

Notebook 04 showed plain CQR under-covers the tails (78% / 81%). The principled fix is **Mondrian
CQR**: a separate conformal correction per band. We implemented and measured it. Does conditioning the
interval on the band fix the tail coverage?

In [ ]:
pb = pd.DataFrame(A["coverage_by_band_plain"])[["band", "coverage", "n"]].rename(columns={"coverage": "plain"})
mb = pd.DataFrame(A["coverage_by_band_mondrian"])[["band", "coverage"]].rename(columns={"coverage": "mondrian"})
cb = pb.merge(mb, on="band")
cb["plain"] = (cb["plain"] * 100).round(0); cb["mondrian"] = (cb["mondrian"] * 100).round(0)
display(cb)

x = np.arange(len(cb))
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.bar(x - 0.2, cb["plain"], width=0.4, label="plain CQR", color="#2c6fbb")
ax.bar(x + 0.2, cb["mondrian"], width=0.4, label="Mondrian CQR", color="#e07b39")
ax.axhline(90, color="k", ls="--", lw=1.2)
ax.set_xticks(x); ax.set_xticklabels(cb["band"], rotation=20)
ax.set(ylabel="Coverage (%)", title="Mondrian CQR ≈ plain CQR per true band (the two bars coincide)", ylim=(60, 105))
ax.legend()
show(save_fig(fig, "limits_mondrian", FIG))

**Reading it — hypothesis 3 CLOSED.** Per-*true*-band coverage is essentially identical under plain and
Mondrian CQR (78/78, 95/96, 98/98, 99/98, 95/95, 80/81). Mondrian conditions on the **predicted**
band — the only choice that is test-time-legal, since the true band is exactly what we lack at
inference — but the point model *shrinks* predictions toward the centre, so games whose true rating is
in an extreme band almost always receive a predicted median in an *interior* band. The extreme
*predicted* bands are nearly empty, leaving Mondrian nothing to recalibrate there. It therefore
improves coverage conditional on the prediction (as guaranteed) but **cannot** improve coverage
conditional on the true band. This is the same single-game noise floor, now proven in interval space.

## The hardest cases: a largest-residual gallery

To make the noise floor concrete, here are the 20 test games the model got most wrong (from the live
error analysis, `reports/largest_residuals.csv`).

In [ ]:
lr = pd.read_csv(REPORTS / "largest_residuals.csv")
lr["game"] = lr["game_id"].str.split("/").str[-1]
view = lr[["game", "rating", "pred", "residual", "n_moves", "cpl_mean", "case"]].copy()
view[["pred", "residual", "cpl_mean"]] = view[["pred", "residual", "cpl_mean"]].round(0)
display(view.head(12))
print("\ncase breakdown:")
print(lr["case"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(6.6, 6))
sc = ax.scatter(lr["rating"], lr["pred"], c=lr["cpl_mean"], cmap="viridis_r", s=70, edgecolor="k", linewidth=0.4)
ax.plot([400, 2800], [400, 2800], "k--", lw=1, label="perfect")
ax.set(xlim=(400, 2800), ylim=(400, 2800), xlabel="Actual rating", ylabel="Predicted rating",
       title="The 20 worst misses (colour = mean CPL)")
ax.legend(loc="upper left")
fig.colorbar(sc, ax=ax, shrink=0.8, label="cpl_mean")
show(save_fig(fig, "limits_worst_cases", FIG))

**Reading it.** The worst misses are not model failures so much as *genuinely ambiguous games*. A
2662-rated player has a scrappy 106-CPL game and is predicted ~1150 — that game really does look like
a weak player's game. Symmetrically, several sub-800 players play one clean, low-CPL game and are
predicted ~2000. The colour (mean CPL) makes it plain: the over-predicted low-raters played
*accurately* that day, and the under-predicted high-raters played *poorly*. No feature set could
recover the truth from these single games, because the single game genuinely resembles a different
rating. This is the noise floor, one row at a time.

## The one lever that works — aggregation

Since the tail error is irreducible *per game*, the only way down is *more games*. Recapping notebook
04's headline: averaging a player's predictions collapses the noise.

In [ ]:
curve = pd.DataFrame(A["aggregation_curve"])
fig, ax = plt.subplots(figsize=(8.2, 4.4))
ax.plot(curve["k"], curve["mae"], "-o", color="#3aa66f", ms=6, lw=2)
ax.axhline(curve["mae"].iloc[0], color="0.6", ls=":", lw=1)
ax.annotate("single game", (curve["k"].iloc[0], curve["mae"].iloc[0]),
            textcoords="offset points", xytext=(20, 4), fontsize=9)
ax.set(xlabel="Games per player (K)", ylabel="MAE (Elo)",
       title="Aggregation: the only lever that moves the floor")
ax.set_ylim(180, 245)
show(save_fig(fig, "limits_aggregation", FIG))
print(f"K=1 MAE {curve['mae'].iloc[0]:.0f}  ->  K=5 MAE {curve[curve['k']==5]['mae'].iloc[0]:.0f}")

## Limitations recap

The honest boundary conditions on everything above:

- **Selection bias.** We model only games with stored evals (~9% of blitz), a non-random subset
  players chose to analyse. Results are calibrated to that population; a random-sample validation with
  self-computed evals is backlogged.
- **Provisional labels.** Lichess strips provisional status from exported PGN, so `drop_provisional`
  is a documented no-op — some labels are noisier (provisional) ratings we cannot filter out.
- **Single-game noise floor.** One blitz game cannot pin a rating; the tail error is the statistically
  optimal hedge under weak evidence, not a fixable bias — shown three independent ways above.
- **Interval guarantee is marginal.** CQR guarantees coverage on average, not per band; the extremes
  under-cover, and no test-time-legal recalibration (Mondrian) can fix it.
- **Scope: blitz only.** Trained and validated on blitz; transfer to rapid/classical is untested.

## Takeaways

- **The tail error is a single-game noise floor, proven three ways:** the calibrating slope is ≈ 1.0
  (already calibrated), added scramble signal leaves the tails at 313/282, and Mondrian CQR cannot
  reach the empty extreme predicted bands.
- **It is a finding, not a bug.** Regression to the mean is the optimal response to weak single-game
  evidence — the worst-miss gallery shows *why* (the games genuinely look like other ratings).
- **Aggregation is the only lever:** ±235 Elo per game → ~200 by K=5. Everything else redistributes
  error; only more games remove it.

This closes the analysis: a calibrated single-game rating estimator with an honest, guaranteed
interval, a clear account of what it cannot do, and the one lever — aggregation — that does.